## **Cosine Similarity and NLP (Natural Language Processing)**

**Natural Language Processing (NLP)** é um campo da AI que estuda como computadores podem analisar, compreender, gerar e interagir com a linguagem humana. Em aplicações de NLP, textos são frequentemente representados como vetores em espaços de alta dimensão. Permitindo aplicar ferramentas matemáticas e algoritmos de ML para analisar relações entre textos.

A **similaridade do cosseno** é uma das métricas mais utilizadas para medir similaridade entre representações vetoriais.

### **Cosine Similarity**

A similaridade do cosseno mede o grau de similaridade entre dois vetores com base no ângulo entre eles:

$$
\text{cosine similarity}(x,y) =
\dfrac{\langle x,y \rangle}{\|x\| \cdot \|y\|}
$$

Sendo especialmente útil quando a direção do vetor é mais relevante que sua magnitude.

### **TF-IDF (Term Frequency e Inverse Document Frequency)**

Para vetorizar os dados após o pré-processamento, vamos usar a técnica TF-IDF. Para isso, considere:

*   Um corpus $D$ com $N$ documentos
*   Um termo $t$
*   Um documento $d$

O peso TF-IDF do termo $t$ no documento $d$ é definido por:

$$
\text{TF-IDF}(t, d, D) = TF(t, d) \cdot IDF(t, D)
$$

#### **TF (Term Frequency)**

$$
TF(t, d) = \dfrac{f_{t, d}}{\displaystyle\sum_{x \in d} f_{x, d}}
$$
onde:

*   $f_{t,d} :=$ número de ocorrências de um termo $t$ em um documento $d$
*   $\displaystyle\sum_{x \in d} f_{x, d} :=$ total de ocorrências de cada termo no documento $d$

#### I**DF (Inverse Document Frequency)**

$$
IDF(t, D) = \log\left(\dfrac{N}{df_{t}}\right)
$$
onde:

*   $N :=$ número total de documentos
*   $df_{t}:=$ número de documentos que contêm o termo $t$

Após vetorizar cada documento, podemos utilizar a similaridade do cosseno.

## **Aplicação simples de NPL**

Dado um JSON contendo frases, o objetivo é encontrar aquelas que são mais semelhantes a uma query.

In [1]:
import json
import pandas as pd

with open("corpus.json") as f:
    data = json.load(f)

df = pd.DataFrame(data, columns=["text"])

print(df)

                                                  text
0    Modern approaches to algebraic geometry combin...
1    Many problems in deep learning require deep th...
2    Persistent homology is widely used in topologi...
3    Optimization problems often rely on calculus t...
4    Modern approaches to theoretical computer scie...
..                                                 ...
105  Formal languages describe sets of strings over...
106  Research in algebraic geometry often relies on...
107  Algebraic geometry studies solutions of polyno...
108      Fundamental groups describe loops in a space.
109  deep learning plays an important role in moder...

[110 rows x 1 columns]


In [2]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words("english"))
ps = PorterStemmer()

def preprocess_text(text):

    # Lowercase
    text = text.lower()

    # Cleaning
    text = re.sub(r"[^\w\s]", "", text)

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords
    tokens = [w for w in tokens if w not in stop_words]

    # Stemming
    tokens = [ps.stem(w) for w in tokens]

    return " ".join(tokens)

In [3]:
df["processed_text"] = df["text"].apply(preprocess_text)

print(df)

                                                  text  \
0    Modern approaches to algebraic geometry combin...   
1    Many problems in deep learning require deep th...   
2    Persistent homology is widely used in topologi...   
3    Optimization problems often rely on calculus t...   
4    Modern approaches to theoretical computer scie...   
..                                                 ...   
105  Formal languages describe sets of strings over...   
106  Research in algebraic geometry often relies on...   
107  Algebraic geometry studies solutions of polyno...   
108      Fundamental groups describe loops in a space.   
109  deep learning plays an important role in moder...   

                                        processed_text  
0    modern approach algebra geometri combin mathem...  
1    mani problem deep learn requir deep theoret un...  
2        persist homolog wide use topolog data analysi  
3            optim problem often reli calculu techniqu  
4    modern approa

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(df["processed_text"])

print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 776 stored elements and shape (110, 151)>
  Coords	Values
  (0, 83)	0.3103233346379326
  (0, 13)	0.4305601593514265
  (0, 7)	0.367180088907839
  (0, 51)	0.4521760941093425
  (0, 26)	0.4305601593514265
  (0, 80)	0.3236883008562082
  (0, 29)	0.2981220764254359
  (1, 79)	0.32676044427474554
  (1, 104)	0.3217298996904001
  (1, 34)	0.5981668872563994
  (1, 74)	0.3552285072470426
  (1, 114)	0.3217298996904001
  (1, 135)	0.30331534226032114
  (1, 143)	0.3319883147436264
  (2, 98)	0.4331151531872365
  (2, 58)	0.39810724376647466
  (2, 150)	0.39810724376647466
  (2, 144)	0.313422320304434
  (2, 138)	0.2651050214229596
  (2, 32)	0.37326873674583533
  (2, 11)	0.4331151531872365
  (3, 104)	0.2761093879730737
  (3, 94)	0.5274562862774861
  (3, 92)	0.32249171882120337
  (3, 112)	0.32249171882120337
  :	:
  (106, 80)	0.2993408650195357
  (106, 92)	0.365583905360412
  (106, 112)	0.365583905360412
  (106, 115)	0.28310991116336953
  (106, 3)	

In [13]:
import numpy as np

query = "I study Topological Spaces"

processed_query = preprocess_text(query)

query_vector = vectorizer.transform([processed_query])

print(query_vector)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3 stored elements and shape (1, 151)>
  Coords	Values
  (0, 127)	0.6357879439094921
  (0, 130)	0.6075159345113523
  (0, 138)	0.4761282177041586


In [14]:
q = query_vector.toarray()[0]

similarities = []

for i in range(X.shape[0]):

    doc = X[i].toarray()[0]

    dot_product = np.dot(q, doc)

    norm_q = np.linalg.norm(q)
    norm_doc = np.linalg.norm(doc)

    cosine = dot_product / (norm_q * norm_doc)

    similarities.append(cosine)

top = np.argsort(similarities)[::-1][:3]

for i in top:
    print("Score:", similarities[i].round(4))
    print("Document:", df["text"][i])
    print()

Score: 0.7269
Document: Algebraic topology studies topological spaces using algebraic tools.

Score: 0.3728
Document: Homology groups capture holes in topological spaces.

Score: 0.3019
Document: Projective spaces are fundamental in algebraic geometry.

